# 02 - GRPO

GRPO generates eight solutions for each problem. Their exact-answer rewards are standardized within each group, and the resulting scalar advantage is applied to every generated token. This notebook explicitly selects classic GRPO because TRL 1.8 defaults to a newer DAPO loss.

In [ ]:
# Hyperparameters
sft_checkpoint = "./checkpoints/sft_base"
dataset_id = "openai/gsm8k"
dataset_config = "main"
dataset_revision = "740312add88f781978c0658806c59bc2815b9866"
seed = 17
train_problems = 256
eval_problems = 64
num_epochs = 1
group_size = 8
unique_prompts_per_update = 16
completions_per_update = unique_prompts_per_update * group_size
train_micro_batch_size = 8
gradient_accumulation_steps = 16
generation_batch_size = 128
generation_micro_batch_size = 16
max_prompt_tokens = 384
max_completion_tokens = 256
temperature = 0.8
top_p = 1.0
learning_rate = 1e-6
weight_decay = 0.0
clip_epsilon = 0.2
reward_std_epsilon = 1e-4
max_grad_norm = 1.0
eval_every_steps = 4
eval_batch_size = 16
wandb_project = "swe-rl-ablations"
wandb_run_name = "grpo"
output_path = "./checkpoints/grpo"

optimizer_steps = train_problems // unique_prompts_per_update
assert completions_per_update == 128
assert train_micro_batch_size * gradient_accumulation_steps == completions_per_update
assert generation_batch_size == completions_per_update
assert train_problems == 256 and eval_problems == 64 and num_epochs == 1
print(f"{unique_prompts_per_update} prompts x {group_size} generations = {completions_per_update} completions/update")

In [ ]:
from pathlib import Path
import os
import re
import time

import torch
import wandb
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import GRPOConfig, GRPOTrainer

project_root = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
checkpoint_path = project_root / sft_checkpoint
if not (checkpoint_path / "config.json").exists():
    raise FileNotFoundError("Run 00_sft.ipynb first.")
os.environ["WANDB_PROJECT"] = wandb_project
set_seed(seed)

In [ ]:
train_rows = load_dataset(dataset_id, dataset_config, split="train", revision=dataset_revision)
test_rows = load_dataset(dataset_id, dataset_config, split="test", revision=dataset_revision)
train_rows = train_rows.shuffle(seed=seed).select(range(train_problems))
eval_rows = test_rows.shuffle(seed=seed).select(range(eval_problems))


def gold_number(answer):
    match = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*$", answer)
    if not match:
        raise ValueError(f"Malformed GSM8K reference answer: {answer!r}")
    return match.group(1).replace(",", "")


def predicted_number(completion):
    if isinstance(completion, list):
        completion = completion[-1]["content"]
    match = re.search(r"Final answer:\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*$", completion.strip(), re.IGNORECASE)
    return None if not match else match.group(1).replace(",", "")


def exact_answer_reward(completions, answer, **_):
    rewards = []
    for completion, reference in zip(completions, answer):
        prediction = predicted_number(completion)
        if prediction is None:
            rewards.append(-1.0)
        else:
            rewards.append(1.0 if float(prediction) == float(gold_number(reference)) else 0.0)
    return rewards


def make_prompt(question):
    return [{"role": "user", "content": f"Solve this grade-school math problem. Show your reasoning, then end with `Final answer: <number>`.\n\n{question}"}]

train_rows = train_rows.map(lambda row: {"prompt": make_prompt(row["question"])})
eval_rows = eval_rows.map(lambda row: {"prompt": make_prompt(row["question"])})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path, use_fast=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

actor = AutoModelForCausalLM.from_pretrained(
    checkpoint_path,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)
reference = AutoModelForCausalLM.from_pretrained(
    checkpoint_path,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
).eval().to("cuda")
for parameter in reference.parameters():
    parameter.requires_grad_(False)

The subclass below keeps TRL's actual sampler, grouped reward normalization, optimizer, and Trainer lifecycle. It only adds chunked generation, independent KL telemetry against the frozen SFT model, clearer metric names, and deterministic exact-answer evaluation.

In [ ]:
class ReadableGRPOTrainer(GRPOTrainer):
    def __init__(self, *args, telemetry_reference, generation_micro_batch_size, exact_eval_rows, **kwargs):
        super().__init__(*args, **kwargs)
        self.telemetry_reference = telemetry_reference
        self.generation_micro_batch_size = generation_micro_batch_size
        self.exact_eval_rows = exact_eval_rows
        self._step_started_at = None

    def _generate_single_turn(self, prompt_ids, images, multimodal_fields):
        completion_ids = []
        logprobs = []
        has_logprobs = True
        for start in range(0, len(prompt_ids), self.generation_micro_batch_size):
            chunk_ids, chunk_logprobs = super()._generate_single_turn(
                prompt_ids[start:start + self.generation_micro_batch_size],
                None if images is None else images[start:start + self.generation_micro_batch_size],
                multimodal_fields,
            )
            completion_ids.extend(chunk_ids)
            if chunk_logprobs is None:
                has_logprobs = False
            else:
                logprobs.extend(chunk_logprobs)
        return completion_ids, logprobs if has_logprobs else None

    def _compute_loss(self, model, inputs):
        started_at = time.perf_counter()
        loss = super()._compute_loss(model, inputs)
        prompt_ids = inputs["prompt_ids"]
        prompt_mask = inputs["prompt_mask"]
        completion_ids = inputs["completion_ids"]
        completion_mask = inputs["completion_mask"]
        input_ids = torch.cat([prompt_ids, completion_ids], dim=1)
        attention_mask = torch.cat([prompt_mask, completion_mask], dim=1)
        with torch.no_grad():
            policy_logps, _, _ = self._get_per_token_logps_and_entropies(
                model, input_ids, attention_mask, completion_ids.size(1), compute_entropy=False
            )
            reference_logits = self.telemetry_reference(
                input_ids=input_ids, attention_mask=attention_mask, use_cache=False
            ).logits[:, -completion_ids.size(1)-1:-1]
            reference_logps = torch.log_softmax(reference_logits.float(), dim=-1).gather(
                -1, completion_ids.unsqueeze(-1)
            ).squeeze(-1)
            kl = ((policy_logps - reference_logps) * completion_mask).sum() / completion_mask.sum().clamp_min(1)
            self._metrics["train"]["kl_sft"].append(float(kl.item()))
            self._metrics["train"]["policy_forward_s"].append(time.perf_counter() - started_at)
        return loss

    def training_step(self, *args, **kwargs):
        torch.cuda.synchronize()
        started_at = time.perf_counter()
        loss = super().training_step(*args, **kwargs)
        torch.cuda.synchronize()
        self._last_policy_step_seconds = time.perf_counter() - started_at
        return loss

    def log(self, logs, *args, **kwargs):
        logs = dict(logs)
        if "reward" in logs:
            logs["train/reward"] = logs["reward"]
        if "loss" in logs:
            logs["loss/policy"] = logs["loss"]
        if "grad_norm" in logs:
            logs["grad_norm/policy"] = logs["grad_norm"]
        if "clip_ratio/region_mean" in logs:
            logs["policy/tokens_clipped_or_masked_pct"] = 100 * logs["clip_ratio/region_mean"]
        if "kl_sft" in logs:
            logs["objective/kl_sft"] = logs["kl_sft"]
        if hasattr(self, "_last_policy_step_seconds"):
            logs["time/policy_forward_backward_s"] = self._last_policy_step_seconds
            logs["time/step_s"] = self._last_policy_step_seconds
        logs["gpu/peak_memory_gb"] = torch.cuda.max_memory_allocated() / 1024**3
        return super().log(logs, *args, **kwargs)

    @torch.no_grad()
    def evaluate(self, *args, **kwargs):
        model = self.accelerator.unwrap_model(self.model)
        model.eval()
        rewards = []
        for start in range(0, len(self.exact_eval_rows), eval_batch_size):
            rows = self.exact_eval_rows.select(range(start, min(start + eval_batch_size, len(self.exact_eval_rows))))
            prompts = [tokenizer.apply_chat_template(make_prompt(question), tokenize=False, add_generation_prompt=True) for question in rows["question"]]
            encoded = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=max_prompt_tokens).to(model.device)
            generated = model.generate(**encoded, do_sample=False, max_new_tokens=max_completion_tokens, pad_token_id=tokenizer.pad_token_id)
            completions = tokenizer.batch_decode(generated[:, encoded.input_ids.shape[1]:], skip_special_tokens=True)
            rewards.extend(exact_answer_reward(completions, rows["answer"]))
        model.train()
        score = sum(rewards) / len(rewards)
        self.log({"eval/reward": score})
        return {"eval/reward": score}


In [ ]:
training_args = GRPOConfig(
    output_dir=str(project_root / output_path),
    run_name=wandb_run_name,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=train_micro_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    generation_batch_size=generation_batch_size,
    steps_per_generation=gradient_accumulation_steps,
    num_generations=group_size,
    num_generations_eval=1,
    max_completion_length=max_completion_tokens,
    temperature=temperature,
    top_p=top_p,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    max_grad_norm=max_grad_norm,
    epsilon=clip_epsilon,
    epsilon_high=clip_epsilon,
    beta=0.0,
    scale_rewards="group",
    loss_type="grpo",
    importance_sampling_level="token",
    eval_strategy="steps",
    eval_steps=eval_every_steps,
    logging_steps=1,
    save_strategy="epoch",
    report_to="wandb",
    bf16=True,
    gradient_checkpointing=True,
)

trainer = ReadableGRPOTrainer(
    model=actor,
    telemetry_reference=reference,
    generation_micro_batch_size=generation_micro_batch_size,
    exact_eval_rows=eval_rows,
    reward_funcs=exact_answer_reward,
    args=training_args,
    train_dataset=train_rows,
    eval_dataset=eval_rows,
    processing_class=tokenizer,
)

trl_log = trainer.log
def log_comparable_metrics(logs, *args, **kwargs):
    if "eval/reward" in logs:
        if wandb.run is not None:
            wandb.log({"eval/reward": logs["eval/reward"]}, step=trainer.state.global_step)
        return
    trl_log(logs, *args, **kwargs)
    comparable = {}
    if "reward" in logs:
        comparable["train/reward"] = logs["reward"]
    if "kl_sft" in logs:
        comparable["objective/kl_sft"] = logs["kl_sft"]
    if "clip_ratio/region_mean" in logs:
        comparable["policy/tokens_clipped_or_masked_pct"] = 100 * logs["clip_ratio/region_mean"]
    for key in ("loss/policy", "grad_norm/policy", "time/step_s", "time/policy_forward_backward_s", "gpu/peak_memory_gb"):
        if key in logs:
            comparable[key] = logs[key]
    if comparable and wandb.run is not None:
        wandb.log(comparable, step=trainer.state.global_step)
trainer.log = log_comparable_metrics

trainer.train()
trainer.save_model(str(project_root / output_path))
tokenizer.save_pretrained(str(project_root / output_path))